In [ ]:
"""Étape 2 : indexation des CG/IPID.
PDF -> texte par page (nettoyé) -> chunks -> embeddings (bge-m3) -> Chroma, avec métadonnées.
Lancer depuis la racine du projet :  python src/indexing.py
"""

In [ ]:
from pypdf import PdfReader             # lire les PDF
from langchain_core.documents import Document   # le format standard d'un texte dans LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter # le découpage
from langchain_ollama import OllamaEmbeddings                   # les embeddings via Ollama
from langchain_chroma import Chroma                                 # la base vectorielle

In [ ]:
def metadonnees_pour_chroma(entree: dict) -> dict:
    """Chroma n'accepte que str/int/float/bool : on convertit les listes et les None."""
    meta = {}
    for cle, valeur in entree.items():
        if valeur is None:
            meta[cle] = ""
        elif isinstance(valeur, list):
            meta[cle] = ", ".join(valeur)
        else:
            meta[cle] = valeur
    return meta


In [ ]:
metadonnees_pour_chroma({"a": 1, "b": ["x", "y"], "c": None})

In [ ]:
from pypdf import PdfReader

reader = PdfReader("../../data/cg/luko_habitation-etendue_cg_2504.pdf")
print("Nombre de pages :", len(reader.pages))

texte = reader.pages[19].extract_text()   # pages[19] = la page 20 du PDF (on compte à partir de 0)
print(texte)

In [ ]:
longueurs = []
for i, page in enumerate(reader.pages):
    texte = page.extract_text() or ""
    longueurs.append((i + 1, len(texte)))      # (numéro de page dans le PDF, nombre de caractères)

# Les 10 pages qui contiennent le moins de texte
for numero, nb in sorted(longueurs, key=lambda x: x[1])[:10]:
    print(f"page {numero:3} : {nb} caractères")

In [ ]:
import re

def nettoyer(texte):
    texte = texte.replace("\u02bc", "'").replace("\u2019", "'")   # 1. apostrophes ʼ et ’ -> '
    texte = texte.replace("\u00a0", " ")                          # 2. espace insécable -> espace normal
    texte = re.sub(r"[ \t]+", " ", texte)                         # 3. plusieurs espaces -> un seul
    texte = re.sub(r"\n\s*\n+", "\n\n", texte)                    # 4. plusieurs lignes vides -> une seule
    return texte.strip()                                          # 5. espaces au début et à la fin

brut = reader.pages[79].extract_text()
propre = nettoyer(brut)

print("AVANT :", repr(brut[:250]))
print()
print("APRÈS :", repr(propre[:250]))

In [ ]:
def metadonnees_pour_chroma(entree: dict) -> dict:
    """Chroma n'accepte que str/int/float/bool : on convertit les listes et les None."""
    meta = {}
    for cle, valeur in entree.items():
        if valeur is None:
            meta[cle] = ""
        elif isinstance(valeur, list):
            meta[cle] = ", ".join(valeur)
        else:
            meta[cle] = valeur
    return meta

In [ ]:
import json

# 1. La fiche d'identité du document, lue dans le catalogue
catalogue = json.load(open("../../data/cg/catalogue.json", encoding="utf-8"))
entree = catalogue[0]                 # la 1re ligne du catalogue = les CG Luko Couverture Étendue
print("Ligne du catalogue :", entree)

base = metadonnees_pour_chroma(entree)
print("Version pour Chroma  :", base)

# 2. Le texte, page par page, avec les métadonnées collées à chaque page
pages = []
for numero, page in enumerate(reader.pages, start=1):
    texte = nettoyer(page.extract_text() or "")
    if len(texte) < 50:
        continue
    pages.append(Document(page_content=texte, metadata={**base, "page": numero}))

print(len(pages), "pages gardées sur", len(reader.pages))
print("Métadonnées de la 1re page gardée :", pages[0].metadata)

In [ ]:
print(pages[0])

In [ ]:
import sys
from pathlib import Path

racine = Path.cwd().parent.parent          # remonte de 2 niveaux, comme le ../../ du PDF
print("Racine du projet :", racine)
print("config.py trouvé ?", (racine / "config.py").exists())

sys.path.insert(0, str(racine))            # Python cherchera d'abord dans la racine

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from config import CHUNK_SIZE, CHUNK_OVERLAP
splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],   # on coupe d'abord entre paragraphes
    )

In [ ]:
chunks = splitter.split_documents(pages)
tous = []
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"{entree['fichier']}#{i}"
    # Affiche le nombre de chunks créés pour chaque document
    # print(f"{entree['fichier']:42} {len(pages):4} pages -> {len(chunks):4} chunks")
    tous.extend(chunks)
print(len(chunks), "chunks créés à partir de", len(pages), "pages")
print("Métadonnées du 1er chunk :", chunks[0].metadata)
print("Texte du 1er chunk :", repr(chunks[0].page_content[:250]))

In [ ]:

import shutil

from config import CHROMA_DIR, COLLECTION, EMBED_MODEL
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)

store = Chroma(
        collection_name=COLLECTION,
        embedding_function=OllamaEmbeddings(model=EMBED_MODEL),
        persist_directory=str(CHROMA_DIR),
        collection_metadata={"hnsw:space": "cosine"},   # distance cosinus
    )

In [ ]:
debut = time.time()
taille_lot = 64
for i in range(0, len(chunks), taille_lot):
    lot = chunks[i:i + taille_lot]
    store.add_documents(lot, ids=[c.metadata["chunk_id"] for c in lot])
    print(f"  embeddings : {min(i + taille_lot, len(chunks))}/{len(chunks)}", end="\r")
print(f"\nIndexation terminée en {time.time() - debut:.0f} s")

In [ ]:
question = "Parlent-ils du vol"

resultats = store.similarity_search_with_score(question, k=3)

for doc, distance in resultats:
    print(f"--- page {doc.metadata['page']} | distance = {distance:.3f}")
    print(doc.page_content[:400])
    print()